# Lev week 3

Цель недели: взять лучшие агрегированные признаки из `lev_week2.ipynb`, добавить к ним лаговое окно в стиле Дениса и перебрать параметры CatBoost на комбинированных признаках.

База из второй недели:

- `extended_agg_window_size = 50`
- `edge_window_size = 10`
- используем все `op_setting_*` и `sensor_*`, включая `sensor_11`
- обязательные агрегированные блоки: `mean`, `std`, `min`, `max`, `range`, `delta`, `slope`, `last_minus_mean`
- лучшие optional-пары: `edge_means`, `energy_acceleration`, `last_first`, `median_iqr`, `relative_changes`
- агрегированная часть: `432` признака

Комбинированный подход: к этим агрегатам добавляем сырое лаговое окно меньшего размера. Сенсоры для лагового окна исключаются как у Дениса: `sensor_19`, `sensor_9`, `sensor_3`, `sensor_1`, `sensor_5`, `sensor_10`, `sensor_7`.


In [36]:
import itertools

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid


In [37]:
columns = [
    "unit_id", "cycle", "op_setting_1", "op_setting_2", "op_setting_3",
    "sensor_1", "sensor_2", "sensor_3", "sensor_4", "sensor_5",
    "sensor_6", "sensor_7", "sensor_8", "sensor_9", "sensor_10",
    "sensor_11", "sensor_12", "sensor_13", "sensor_14", "sensor_15",
    "sensor_16", "sensor_17", "sensor_18", "sensor_19", "sensor_20", "sensor_21",
]

train_df = pd.read_csv("CMAPSSData/train_FD001.txt", sep=r"\s+", header=None, names=columns)
test_df = pd.read_csv("CMAPSSData/test_FD001.txt", sep=r"\s+", header=None, names=columns)
rul_test_df = pd.read_csv("CMAPSSData/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

target_col = "Remaining Useful Life"
train_df[target_col] = train_df["unit_id"].map(train_df.groupby("unit_id")["cycle"].max()) - train_df["cycle"]

feature_cols = [
    col for col in train_df.columns
    if col.startswith("op_setting_") or col.startswith("sensor_")
]


In [38]:
extended_agg_window_size = 50
edge_window_size = 10
extended_agg_feature_cols = feature_cols
eps = 1e-8

mandatory_feature_blocks = [
    "mean",
    "std",
    "min",
    "max",
    "range",
    "delta",
    "slope",
    "last_minus_mean",
]

optional_feature_pairs = {
    "median_iqr": ["median", "iqr"],
    "last_first": ["last", "first"],
    "relative_changes": ["relative_delta", "relative_last_minus_mean"],
    "edge_means": ["mean_last_10", "mean_first_10"],
    "edge_change_volatility": ["mean_last_10_minus_first_10", "std_last_10"],
    "tail_quantiles": ["q10", "q90"],
    "robust_spread": ["q90_q10_range", "mad"],
    "energy_acceleration": ["rms", "slope_change"],
}

best_week2_optional_pairs = [
    "edge_means",
    "energy_acceleration",
    "last_first",
    "median_iqr",
    "relative_changes",
]

best_week2_optional_blocks = list(itertools.chain.from_iterable(
    optional_feature_pairs[pair_name]
    for pair_name in best_week2_optional_pairs
))
best_week2_blocks = mandatory_feature_blocks + best_week2_optional_blocks



In [39]:
def calculate_slope(values):
    x = np.arange(len(values))
    x = x - x.mean()
    denominator = (x ** 2).sum()
    if denominator == 0:
        return 0
    return np.dot(values, x) / denominator


def calculate_first_edge_mean(values):
    return values[:edge_window_size].mean()


def calculate_mad(values):
    median_value = np.median(values)
    return np.median(np.abs(values - median_value))


def calculate_rms(values):
    return np.sqrt(np.mean(values ** 2))


def calculate_slope_change(values):
    split_index = len(values) // 2
    if split_index == 0 or split_index == len(values):
        return 0
    return calculate_slope(values[split_index:]) - calculate_slope(values[:split_index])


def rename_feature_block(block, suffix):
    renamed_block = block.copy()
    renamed_block.columns = [
        f"{col}_{suffix}_{extended_agg_window_size}"
        for col in renamed_block.columns
    ]
    return renamed_block


In [40]:
def build_all_agg_window_features(source_df, include_target=False, require_full_window=False):
    sorted_df = source_df.sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    min_periods = extended_agg_window_size if require_full_window else 1
    edge_min_periods = edge_window_size if require_full_window else 1

    rolling_features = sorted_df.groupby("unit_id")[extended_agg_feature_cols].rolling(
        window=extended_agg_window_size,
        min_periods=min_periods,
    )
    edge_rolling_features = sorted_df.groupby("unit_id")[extended_agg_feature_cols].rolling(
        window=edge_window_size,
        min_periods=edge_min_periods,
    )

    mean_features = rolling_features.mean().reset_index(level=0, drop=True)
    std_features = rolling_features.std(ddof=0).reset_index(level=0, drop=True).fillna(0)
    min_features = rolling_features.min().reset_index(level=0, drop=True)
    max_features = rolling_features.max().reset_index(level=0, drop=True)
    median_features = rolling_features.median().reset_index(level=0, drop=True)
    q10_features = rolling_features.quantile(0.10).reset_index(level=0, drop=True)
    q25_features = rolling_features.quantile(0.25).reset_index(level=0, drop=True)
    q75_features = rolling_features.quantile(0.75).reset_index(level=0, drop=True)
    q90_features = rolling_features.quantile(0.90).reset_index(level=0, drop=True)

    range_features = max_features - min_features
    iqr_features = q75_features - q25_features
    q90_q10_range_features = q90_features - q10_features

    first_features = sorted_df.groupby("unit_id")[extended_agg_feature_cols].shift(extended_agg_window_size - 1)
    if not require_full_window:
        first_available_values = sorted_df.groupby("unit_id")[extended_agg_feature_cols].transform("first")
        first_features = first_features.fillna(first_available_values)

    last_features = sorted_df[extended_agg_feature_cols]
    delta_features = last_features - first_features
    last_minus_mean_features = last_features - mean_features
    relative_delta_features = delta_features / (first_features.abs() + eps)
    relative_last_minus_mean_features = last_minus_mean_features / (mean_features.abs() + eps)

    mean_last_10_features = edge_rolling_features.mean().reset_index(level=0, drop=True)
    std_last_10_features = edge_rolling_features.std(ddof=0).reset_index(level=0, drop=True).fillna(0)
    mean_first_10_features = rolling_features.apply(
        calculate_first_edge_mean,
        raw=True,
    ).reset_index(level=0, drop=True)
    mean_last_10_minus_first_10_features = mean_last_10_features - mean_first_10_features

    slope_features = rolling_features.apply(
        calculate_slope,
        raw=True,
    ).reset_index(level=0, drop=True)
    mad_features = rolling_features.apply(
        calculate_mad,
        raw=True,
    ).reset_index(level=0, drop=True)
    rms_features = rolling_features.apply(
        calculate_rms,
        raw=True,
    ).reset_index(level=0, drop=True)
    slope_change_features = rolling_features.apply(
        calculate_slope_change,
        raw=True,
    ).reset_index(level=0, drop=True)

    feature_blocks = {
        "mean": mean_features,
        "std": std_features,
        "min": min_features,
        "max": max_features,
        "range": range_features,
        "delta": delta_features,
        "slope": slope_features,
        "last_minus_mean": last_minus_mean_features,
        "median": median_features,
        "iqr": iqr_features,
        "last": last_features,
        "first": first_features,
        "relative_delta": relative_delta_features,
        "relative_last_minus_mean": relative_last_minus_mean_features,
        "mean_last_10": mean_last_10_features,
        "mean_first_10": mean_first_10_features,
        "mean_last_10_minus_first_10": mean_last_10_minus_first_10_features,
        "std_last_10": std_last_10_features,
        "q10": q10_features,
        "q90": q90_features,
        "q90_q10_range": q90_q10_range_features,
        "mad": mad_features,
        "rms": rms_features,
        "slope_change": slope_change_features,
    }

    metadata_cols = ["unit_id", "cycle"]
    if include_target:
        metadata_cols.append(target_col)

    renamed_feature_blocks = {
        suffix: rename_feature_block(block, suffix)
        for suffix, block in feature_blocks.items()
    }

    result_df = pd.concat(
        [sorted_df[metadata_cols], *renamed_feature_blocks.values()],
        axis=1,
    )

    if require_full_window:
        result_df = result_df[result_df["cycle"] >= extended_agg_window_size].dropna()

    result_df = result_df.reset_index(drop=True)
    feature_block_columns = {
        suffix: list(block.columns)
        for suffix, block in renamed_feature_blocks.items()
    }

    return result_df, feature_block_columns


In [41]:
print("Building train features...")
all_agg_window_train_df, feature_block_columns = build_all_agg_window_features(
    train_df,
    include_target=True,
    require_full_window=True,
)
print("Building test features...")
all_agg_window_test_df, _ = build_all_agg_window_features(
    test_df,
    include_target=False,
    require_full_window=False,
)

all_agg_window_test_last_rows = (
    all_agg_window_test_df
    .sort_values(["unit_id", "cycle"])
    .groupby("unit_id")
    .tail(1)
    .sort_values("unit_id")
    .reset_index(drop=True)
)

def feature_columns_for_blocks(block_names):
    selected_columns = []
    for block_name in block_names:
        selected_columns.extend(feature_block_columns[block_name])
    return selected_columns

best_week2_feature_cols = feature_columns_for_blocks(best_week2_blocks)

X_train_best_week2 = all_agg_window_train_df[best_week2_feature_cols]
y_train_best_week2 = all_agg_window_train_df[target_col].reset_index(drop=True)
X_test_best_week2 = all_agg_window_test_last_rows[best_week2_feature_cols]
y_test_best_week2 = rul_test_df["RUL"].reset_index(drop=True)

if len(X_test_best_week2) != len(y_test_best_week2):
    raise ValueError("test feature rows must match RUL target rows")

print("Features are ready")


Building train features...
Building test features...
Features are ready


## Combined aggregate + lag features

Здесь строим несколько наборов признаков: агрегаты week2 плюс лаговое окно разного размера. Окно `30` повторяет идею Дениса, окна `10` и `20` проверяют более короткую динамику.


In [42]:
denis_excluded_sensors_by_number = [19, 9, 3, 1, 5, 10, 7]
combined_lag_feature_configs = [
    {
        "feature_config": "agg_plus_lag_10_denis_sensors",
        "lag_window_size": 10,
        "excluded_sensors_by_number": denis_excluded_sensors_by_number,
    },
    {
        "feature_config": "agg_plus_lag_20_denis_sensors",
        "lag_window_size": 20,
        "excluded_sensors_by_number": denis_excluded_sensors_by_number,
    },
    {
        "feature_config": "agg_plus_lag_30_denis_sensors",
        "lag_window_size": 30,
        "excluded_sensors_by_number": denis_excluded_sensors_by_number,
    },
]


def selected_lag_feature_cols(excluded_sensors_by_number):
    excluded_sensor_cols = {
        f"sensor_{int(sensor_num)}"
        for sensor_num in excluded_sensors_by_number
    }
    excluded_sensor_cols = sorted(col for col in excluded_sensor_cols if col in feature_cols)
    selected_cols = [col for col in feature_cols if col not in excluded_sensor_cols]
    if not selected_cols:
        raise ValueError("No lag features left after sensor exclusions")
    return selected_cols, excluded_sensor_cols


def build_lag_window_features(source_df, selected_feature_cols, lag_window_size, include_target=False):
    sorted_df = source_df.sort_values(["unit_id", "cycle"]).reset_index(drop=True)
    first_values = sorted_df.groupby("unit_id")[selected_feature_cols].transform("first")
    lagged_parts = []

    for lag in range(lag_window_size):
        shifted = sorted_df.groupby("unit_id")[selected_feature_cols].shift(lag)
        shifted = shifted.fillna(first_values)
        suffix = "t" if lag == 0 else f"t_minus_{lag}"
        shifted.columns = [f"{col}_{suffix}" for col in selected_feature_cols]
        lagged_parts.append(shifted)

    metadata_cols = ["unit_id", "cycle"]
    if include_target:
        metadata_cols.append(target_col)

    return pd.concat([sorted_df[metadata_cols], *lagged_parts], axis=1).reset_index(drop=True)


def build_combined_feature_set(config):
    lag_window_size = config["lag_window_size"]
    lag_base_cols, excluded_sensor_cols = selected_lag_feature_cols(
        config["excluded_sensors_by_number"]
    )

    print(f"Building lag features for {config['feature_config']}...")
    lag_train_df = build_lag_window_features(
        train_df,
        selected_feature_cols=lag_base_cols,
        lag_window_size=lag_window_size,
        include_target=False,
    )
    lag_test_df = build_lag_window_features(
        test_df,
        selected_feature_cols=lag_base_cols,
        lag_window_size=lag_window_size,
        include_target=False,
    )

    lag_feature_cols = [
        col for col in lag_train_df.columns
        if col not in ["unit_id", "cycle", target_col]
    ]

    train_lag_aligned = all_agg_window_train_df[["unit_id", "cycle"]].merge(
        lag_train_df[["unit_id", "cycle", *lag_feature_cols]],
        on=["unit_id", "cycle"],
        how="left",
        validate="one_to_one",
    )
    test_lag_last_rows = (
        lag_test_df
        .sort_values(["unit_id", "cycle"])
        .groupby("unit_id")
        .tail(1)
        .sort_values("unit_id")
        .reset_index(drop=True)
    )

    if train_lag_aligned[lag_feature_cols].isna().any().any():
        raise ValueError(f"Lag train features contain missing values for {config['feature_config']}")
    if len(test_lag_last_rows) != len(y_test_best_week2):
        raise ValueError(f"Lag test rows mismatch for {config['feature_config']}")

    X_train_combined = pd.concat(
        [
            X_train_best_week2.reset_index(drop=True),
            train_lag_aligned[lag_feature_cols].reset_index(drop=True),
        ],
        axis=1,
    )
    X_test_combined = pd.concat(
        [
            X_test_best_week2.reset_index(drop=True),
            test_lag_last_rows[lag_feature_cols].reset_index(drop=True),
        ],
        axis=1,
    )

    print(
        f"{config['feature_config']}: "
        f"agg_features={X_train_best_week2.shape[1]}, "
        f"lag_features={len(lag_feature_cols)}, "
        f"total_features={X_train_combined.shape[1]}, "
        f"excluded_lag_sensors={excluded_sensor_cols}"
    )

    return {
        "feature_config": config["feature_config"],
        "lag_window_size": lag_window_size,
        "excluded_lag_sensors": excluded_sensor_cols,
        "lag_feature_count": len(lag_feature_cols),
        "total_feature_count": X_train_combined.shape[1],
        "X_train": X_train_combined,
        "X_test": X_test_combined,
        "y_train": y_train_best_week2,
        "y_test": y_test_best_week2,
    }


combined_feature_sets = [
    build_combined_feature_set(config)
    for config in combined_lag_feature_configs
]
print(f"Combined feature sets ready: {len(combined_feature_sets)}")


Building lag features for agg_plus_lag_10_denis_sensors...
agg_plus_lag_10_denis_sensors: agg_features=432, lag_features=170, total_features=602, excluded_lag_sensors=['sensor_1', 'sensor_10', 'sensor_19', 'sensor_3', 'sensor_5', 'sensor_7', 'sensor_9']
Building lag features for agg_plus_lag_20_denis_sensors...
agg_plus_lag_20_denis_sensors: agg_features=432, lag_features=340, total_features=772, excluded_lag_sensors=['sensor_1', 'sensor_10', 'sensor_19', 'sensor_3', 'sensor_5', 'sensor_7', 'sensor_9']
Building lag features for agg_plus_lag_30_denis_sensors...
agg_plus_lag_30_denis_sensors: agg_features=432, lag_features=510, total_features=942, excluded_lag_sensors=['sensor_1', 'sensor_10', 'sensor_19', 'sensor_3', 'sensor_5', 'sensor_7', 'sensor_9']
Combined feature sets ready: 3


## CatBoost parameter search

Перебираем все комбинации из `catboost_param_grid` для каждого комбинированного набора признаков. Во время работы каждая модель печатает номер запуска `i/N`, название feature-конфига и параметры CatBoost.

Перед запуском выбери, где учить CatBoost:

- `catboost_task_type = "CPU"` - обучение на процессоре
- `catboost_task_type = "GPU"` - обучение на видеокарте


In [43]:
catboost_task_type = "CPU"  # "CPU" или "GPU"
catboost_devices = "0"      # номер видеокарты, используется только для GPU

if catboost_task_type not in {"CPU", "GPU"}:
    raise ValueError('catboost_task_type must be "CPU" or "GPU"')

catboost_param_grid = {
    "iterations": [500, 1000, 1500],
    "learning_rate": [0.02, 0.03, 0.05],
    "depth": [4, 6, 8],
    "l2_leaf_reg": [1.0, 3.0, 5.0],
}

fixed_catboost_params = {
    "loss_function": "RMSE",
    "eval_metric": "MAE",
    "random_seed": 42,
    "verbose": 200,
    "allow_writing_files": False,
    "task_type": catboost_task_type,
}

if catboost_task_type == "GPU":
    fixed_catboost_params["devices"] = catboost_devices

catboost_grid = list(ParameterGrid(catboost_param_grid))
print(f"CatBoost task_type: {catboost_task_type}")
if catboost_task_type == "GPU":
    print(f"CatBoost GPU devices: {catboost_devices}")
total_catboost_runs = len(combined_feature_sets) * len(catboost_grid)
print(f"Feature configs to test: {len(combined_feature_sets)}")
print(f"CatBoost params per feature config: {len(catboost_grid)}")
print(f"Total CatBoost models to train: {total_catboost_runs}")


CatBoost task_type: CPU
Feature configs to test: 3
CatBoost params per feature config: 81
Total CatBoost models to train: 243


In [44]:
catboost_results = []

def evaluate_catboost_params(feature_set, params, model_number, total_models):
    feature_config = feature_set["feature_config"]
    print(
        f"Training CatBoost {model_number}/{total_models} on {catboost_task_type}: "
        f"feature_config={feature_config}, params={params}"
    )
    model = CatBoostRegressor(
        **fixed_catboost_params,
        **params,
    )
    model.fit(feature_set["X_train"], feature_set["y_train"])

    y_pred = model.predict(feature_set["X_test"])
    result = {
        "feature_config": feature_config,
        "lag_window_size": feature_set["lag_window_size"],
        "lag_feature_count": feature_set["lag_feature_count"],
        "total_feature_count": feature_set["total_feature_count"],
        **params,
        "mae": mean_absolute_error(feature_set["y_test"], y_pred),
        "rmse": mean_squared_error(feature_set["y_test"], y_pred) ** 0.5,
        "r2": r2_score(feature_set["y_test"], y_pred),
    }
    print(
        f"Finished CatBoost {model_number}/{total_models}: "
        f"feature_config={feature_config}, "
        f"MAE={result['mae']:.6f}, RMSE={result['rmse']:.6f}, R2={result['r2']:.6f}"
    )
    return result

model_number = 0
for feature_set in combined_feature_sets:
    for params in catboost_grid:
        model_number += 1
        catboost_results.append(
            evaluate_catboost_params(
                feature_set=feature_set,
                params=params,
                model_number=model_number,
                total_models=total_catboost_runs,
            )
        )

catboost_results_df = pd.DataFrame(catboost_results).sort_values("mae").reset_index(drop=True)
catboost_results_df


Training CatBoost 1/243 on CPU: feature_config=agg_plus_lag_10_denis_sensors, params={'depth': 4, 'iterations': 500, 'l2_leaf_reg': 1.0, 'learning_rate': 0.02}
0:	learn: 45.6899829	total: 3.9ms	remaining: 1.94s
200:	learn: 14.7024825	total: 629ms	remaining: 935ms
400:	learn: 12.0891922	total: 1.2s	remaining: 297ms
499:	learn: 11.1408394	total: 1.49s	remaining: 0us
Finished CatBoost 1/243: feature_config=agg_plus_lag_10_denis_sensors, MAE=15.820852, RMSE=22.268274, R2=0.712847
Training CatBoost 2/243 on CPU: feature_config=agg_plus_lag_10_denis_sensors, params={'depth': 4, 'iterations': 500, 'l2_leaf_reg': 1.0, 'learning_rate': 0.03}
0:	learn: 45.3507666	total: 3.91ms	remaining: 1.95s
200:	learn: 13.2407855	total: 615ms	remaining: 914ms
400:	learn: 10.3261370	total: 1.19s	remaining: 294ms
499:	learn: 9.2570567	total: 1.48s	remaining: 0us
Finished CatBoost 2/243: feature_config=agg_plus_lag_10_denis_sensors, MAE=15.578590, RMSE=22.182673, R2=0.715050
Training CatBoost 3/243 on CPU: featu

,feature_config,lag_window_size,lag_feature_count,total_feature_count,depth,iterations,l2_leaf_reg,learning_rate,mae,rmse,r2
0,agg_plus_lag_10_denis_sensors,10,170,602,8,1500,3.0,0.03,13.494217,19.992126,0.768549
1,agg_plus_lag_10_denis_sensors,10,170,602,8,1500,5.0,0.03,13.515956,19.888779,0.770936
2,agg_plus_lag_10_denis_sensors,10,170,602,8,1000,3.0,0.03,13.537079,20.062345,0.766921
3,agg_plus_lag_10_denis_sensors,10,170,602,8,1000,5.0,0.03,13.574006,19.987124,0.768665
4,agg_plus_lag_10_denis_sensors,10,170,602,6,1500,3.0,0.05,13.650248,20.601485,0.754225
...,...,...,...,...,...,...,...,...,...,...,...
238,agg_plus_lag_10_denis_sensors,10,170,602,4,500,3.0,0.02,15.739016,22.179352,0.715136
239,agg_plus_lag_10_denis_sensors,10,170,602,4,500,1.0,0.02,15.820852,22.268274,0.712847
240,agg_plus_lag_30_denis_sensors,30,510,942,4,500,3.0,0.02,15.868231,22.052396,0.718387
241,agg_plus_lag_10_denis_sensors,10,170,602,4,500,5.0,0.03,15.931709,22.703902,0.701502


In [45]:
best_catboost_result = catboost_results_df.iloc[0]

print("Best combined CatBoost result")
print(f"feature_config: {best_catboost_result['feature_config']}")
print(f"lag_window_size: {best_catboost_result['lag_window_size']}")
print(f"total_feature_count: {best_catboost_result['total_feature_count']}")
print(f"iterations: {best_catboost_result['iterations']}")
print(f"learning_rate: {best_catboost_result['learning_rate']}")
print(f"depth: {best_catboost_result['depth']}")
print(f"l2_leaf_reg: {best_catboost_result['l2_leaf_reg']}")
print(f"MAE: {best_catboost_result['mae']:.6f}")
print(f"RMSE: {best_catboost_result['rmse']:.6f}")
print(f"R2: {best_catboost_result['r2']:.6f}")


Best combined CatBoost result
feature_config: agg_plus_lag_10_denis_sensors
lag_window_size: 10
total_feature_count: 602
iterations: 1500
learning_rate: 0.03
depth: 8
l2_leaf_reg: 3.0
MAE: 13.494217
RMSE: 19.992126
R2: 0.768549
